In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("powerplant_data.csv")

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
X = df.drop("PE", axis = 1)
Y = df["PE"]

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size = 0.3, random_state = 42
)

In [7]:
df.shape

(9568, 5)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
X_test_scaled

array([[ 1.35027377,  0.23835089, -1.2868797 , -1.1152662 ],
       [ 0.81612819,  1.36219539, -0.74354734,  0.25518223],
       [-0.23877586, -0.73921294,  1.9883008 , -0.20689783],
       ...,
       [ 0.23512774,  0.27059555,  0.93538342, -0.23848131],
       [-1.79971257, -1.42106962,  2.6025026 ,  1.16698358],
       [ 1.16285427,  0.78729663, -0.34195385, -0.32361939]])

In [10]:
type(X_train_scaled)

numpy.ndarray

In [11]:
type(Y_train)

pandas.Series

In [12]:
print(X_train_scaled.shape)

(6697, 4)


In [13]:
print(Y_train.shape)

(6697,)


In [14]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)

Y_train_tensor = torch.tensor(Y_train.values, dtype = torch.float32).view(-1, 1)
Y_test_tensor = torch.tensor(Y_test.values, dtype = torch.float32).view(-1, 1)

In [15]:
#the datatype above X_train_scaled is float, hence dtype is float32

#upar datatype of X_train_scaled is numpy array, issliee we can access it directly
#but Y_train ka is pandas series, so to access values we use .values

#the shape of X_train_scaled is (n, m), but that of Y_train is (n)
#so to convert it from 1 array to the required format, we used .views(-1, 1)

In [16]:
#Dataloader is used to define how data will be loaded for training
#TensorDataset is used to define data. it combines multiples tensors into single datasheet

In [17]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,  Y_test_tensor)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = 32)

DEEP LEARNING

In [19]:
#Define an ANN model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #1st hidden layer
            #nn.Linear(input_features, output_features)
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            #2nd hidden layer
            nn.Linear(6, 6),
            nn.ReLU(),

            #output layer
            nn.Linear(6, 1),
        )


    def forward(self, x):
        return self.model(x)

In [20]:
import torch.optim as optim

model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [21]:
#Training
epochs = 100
train_losses = []
val_losses = []

best_val_loss = float("inf")

for epoch in range (epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        # xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()   #it is used to adjust the gradients

        outputs = model(xb)     #forwards propagation 
        loss = criterion(outputs, yb)    # use to calculate loss between current trained batch and already existing labels of that batch.
        loss.backward()        #backward propagation ... compute gradients
        optimizer.step()       #params update

        running_loss += loss.item()   #to convert pytorch tensor value to python, we use .items()

        epoch_train_loss = running_loss / len(train_loader)
        train_losses.append(epoch_train_loss)

    #Validation
    #validation mai we don't do backward prop as no weight readjustment etc is needed.
    model.eval()          #we need to add this statement to show that we have started evaluation
    running_val_loss = 0.0

    #by default pytorch mai auto grad hota hai jisme it automatically calculates gradient, so yaha gradient calculte na karneko kaha gya hai (no backward propagation)
    with torch.no_grad():  
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> train loss = {epoch_train_loss} & val loss = {epoch_val_loss}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

epoch 1/100 ==> train loss = 205893.61473214286 & val loss = 203534.91770833332
epoch 2/100 ==> train loss = 196166.03117559524 & val loss = 183740.91302083334
epoch 3/100 ==> train loss = 163673.23199404762 & val loss = 138759.82274305556
epoch 4/100 ==> train loss = 114956.28240327381 & val loss = 90561.46675347222
epoch 5/100 ==> train loss = 73617.4515811012 & val loss = 56543.108723958336
epoch 6/100 ==> train loss = 44378.745796130956 & val loss = 32110.116319444445
epoch 7/100 ==> train loss = 23889.765973772322 & val loss = 16483.256184895832
epoch 8/100 ==> train loss = 12504.852822730654 & val loss = 9207.351302083332
epoch 9/100 ==> train loss = 7753.268645368304 & val loss = 6260.126814778646
epoch 10/100 ==> train loss = 5558.191500418527 & val loss = 4648.106431749132
epoch 11/100 ==> train loss = 4209.872798084078 & val loss = 3536.5957478841146
epoch 12/100 ==> train loss = 3213.4036493210565 & val loss = 2692.90638156467
epoch 13/100 ==> train loss = 2462.635746837798 

In [22]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "Validation Loss": val_losses
})

plt.plot(loss_df["Training Loss"], label = "Training Loss")
plt.plot(loss_df["Validation Loss"], label = "Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

ValueError: All arrays must be of the same length

In [23]:
#When we are judging our model performance we must do it based on model's validatiom loss. becoz it shows unknown data pe kaisa performance hai.
model.load_state_dict(torch.load("best_model.pt"))

#best model . pt se hum sare weights k values lete hai, torch.load se we get the raw values and then , we load it to the model using load_state

<All keys matched successfully>

In [24]:
#Evaluation

model.eval()
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criterion(train_preds, Y_train_tensor)
    test_mse_loss = criterion(test_preds, Y_test_tensor)

print("Training MSE: ", train_mse_loss.item())
print("Testing MSE: ", test_mse_loss.item())

Training MSE:  20.480268478393555
Testing MSE:  19.17917251586914


In [25]:
from sklearn.metrics import r2_score

print("r2 score =", r2_score(Y_test, test_preds))

r2 score = 0.9333980244640051


In [28]:
predicted_df = pd.DataFrame(test_preds.numpy(),  columns = ["Predicted Values"])
actual_df = pd.DataFrame(Y_test.values,  columns = ["Actual Values"])

pd.concat([predicted_df, actual_df], axis = 1)

,Predicted Values,Actual Values
0,435.189178,433.27
1,436.780212,438.16
2,461.547302,458.42
3,476.682800,480.82
4,435.035461,441.41
...,...,...
2866,480.545776,480.31
2867,452.774567,446.77
2868,450.761719,454.66
2869,484.230042,483.77


In [1]:
type(test_preds)

NameError: name 'test_preds' is not defined